In [66]:
import warnings
warnings.filterwarnings("ignore")

In [67]:
import datasets
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf

In [68]:
from datasets import load_dataset
import pandas as pd
ds=load_dataset("Tobi-Bueck/customer-support-tickets")

In [69]:
df=pd.DataFrame(ds['train'])
df.to_csv("data.csv")

In [70]:
df=df[['body','queue','language']]
df.head()

,body,queue,language
0,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Technical Support,de
1,"Dear Customer Support Team,\n\nI am writing to...",Technical Support,en
2,"Dear Customer Support Team,\n\nI hope this mes...",Returns and Exchanges,en
3,"Dear Customer Support Team,\n\nI hope this mes...",Billing and Payments,en
4,"Dear Support Team,\n\nI hope this message reac...",Sales and Pre-Sales,en


In [77]:
df['body'].isnull().sum()
df['body']=df['body'].fillna("")
df['body']=df['body'].fillna("").str.lower()

In [78]:
import re

df['body']=df['body'].apply(lambda x: re.sub(r'nn+',' ',x))
df['body']=df['body'].apply(lambda x: re.sub(r'\s+', ' ',x).strip())
df['body']=df['body'].apply(lambda x: re.sub(r'([a-z])([A-Z])',r'\1 \2',x))

In [79]:
import re
import string

df['body']=df['body'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

In [74]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [80]:
from nltk.tokenize import word_tokenize
df['tokens']=df['body'].apply(word_tokenize)
df.head()

,body,queue,language,tokens
0,sehr geehrtes supportteam ich möchte einen gra...,Technical Support,de,"[sehr, geehrtes, supportteam, ich, möchte, ein..."
1,dear customer support team i am writing to rep...,Technical Support,en,"[dear, customer, support, team, i, am, writing..."
2,dear customer support team i hope this message...,Returns and Exchanges,en,"[dear, customer, support, team, i, hope, this,..."
3,dear customer support team i hope this message...,Billing and Payments,en,"[dear, customer, support, team, i, hope, this,..."
4,dear support team i hope this message reaches ...,Sales and Pre-Sales,en,"[dear, support, team, i, hope, this, message, ..."


In [81]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

english=set(stopwords.words('english'))
german=set(stopwords.words('german'))

df['tokens']=df['tokens'].apply(
    lambda words:[w.lower() for w in words if w.isalpha()]
)
df.loc[df['language'] == 'en', 'tokens'] = df.loc[df['language'] == 'en', 'tokens'].apply(
    lambda words: [w for w in words if w not in english]
)

df.loc[df['language'] == 'de', 'tokens'] = df.loc[df['language'] == 'de', 'tokens'].apply(
    lambda words: [w for w in words if w not in german]
)

df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,body,queue,language,tokens
0,sehr geehrtes supportteam ich möchte einen gra...,Technical Support,de,"[geehrtes, supportteam, möchte, gravierenden, ..."
1,dear customer support team i am writing to rep...,Technical Support,en,"[dear, customer, support, team, writing, repor..."
2,dear customer support team i hope this message...,Returns and Exchanges,en,"[dear, customer, support, team, hope, message,..."
3,dear customer support team i hope this message...,Billing and Payments,en,"[dear, customer, support, team, hope, message,..."
4,dear support team i hope this message reaches ...,Sales and Pre-Sales,en,"[dear, support, team, hope, message, reaches, ..."


In [82]:
df['tokens']=df['tokens'].apply(lambda words:[w.lower() for w in words])
df.head()

,body,queue,language,tokens
0,sehr geehrtes supportteam ich möchte einen gra...,Technical Support,de,"[geehrtes, supportteam, möchte, gravierenden, ..."
1,dear customer support team i am writing to rep...,Technical Support,en,"[dear, customer, support, team, writing, repor..."
2,dear customer support team i hope this message...,Returns and Exchanges,en,"[dear, customer, support, team, hope, message,..."
3,dear customer support team i hope this message...,Billing and Payments,en,"[dear, customer, support, team, hope, message,..."
4,dear support team i hope this message reaches ...,Sales and Pre-Sales,en,"[dear, support, team, hope, message, reaches, ..."


In [83]:
df['tokens']=df['tokens'].apply(lambda words: [w for w in words if w.isalpha()])
df['tokens'].head()

0    [geehrtes, supportteam, möchte, gravierenden, ...
1    [dear, customer, support, team, writing, repor...
2    [dear, customer, support, team, hope, message,...
3    [dear, customer, support, team, hope, message,...
4    [dear, support, team, hope, message, reaches, ...
Name: tokens, dtype: object

In [84]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [85]:
from nltk.stem import WordNetLemmatizer

lemma=WordNetLemmatizer()
df['tokens']=df['tokens'].apply(
    lambda words:[lemma.lemmatize(w) for w in words]
)
df.head()

,body,queue,language,tokens
0,sehr geehrtes supportteam ich möchte einen gra...,Technical Support,de,"[geehrtes, supportteam, möchte, gravierenden, ..."
1,dear customer support team i am writing to rep...,Technical Support,en,"[dear, customer, support, team, writing, repor..."
2,dear customer support team i hope this message...,Returns and Exchanges,en,"[dear, customer, support, team, hope, message,..."
3,dear customer support team i hope this message...,Billing and Payments,en,"[dear, customer, support, team, hope, message,..."
4,dear support team i hope this message reaches ...,Sales and Pre-Sales,en,"[dear, support, team, hope, message, reach, we..."


In [86]:
df['tokens']=df['tokens'].apply(lambda words: " ".join(words))

In [87]:
print(df.columns)

Index(['body', 'queue', 'language', 'tokens'], dtype='object')


In [88]:
x=df['tokens']
y=df['queue']

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [89]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(max_features=3000)
x_train_vec=tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [53]:
tfidf = TfidfVectorizer(
    max_features=1500,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95
)
x_train_vec=tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [ ]:
df.to_csv("clean_data.csv")

In [ ]:
df = ds['train'].to_pandas()

In [90]:
from sklearn.linear_model import LogisticRegression
m1=LogisticRegression(max_iter=1000,class_weight='balanced')
m1.fit(x_train_vec,y_train)
y_pred=m1.predict(X_test_vec)

In [91]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.3921314660406379
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.76      0.65      0.70        54
            Arts & Entertainment/Music       0.52      0.60      0.56        77
          Autos & Vehicles/Maintenance       0.36      0.48      0.41        62
                Autos & Vehicles/Sales       0.40      0.47      0.43        66
            Beauty & Fitness/Cosmetics       0.63      0.76      0.69        55
     Beauty & Fitness/Fitness Training       0.57      0.58      0.58        60
                  Billing and Payments       0.72      0.66      0.69       990
            Books & Literature/Fiction       0.52      0.63      0.57        52
        Books & Literature/Non-Fiction       0.66      0.59      0.62        63
   Business & Industrial/Manufacturing       0.66      0.58      0.61        73
                      Customer Service       0.34      0.21      0

In [92]:
from sklearn.naive_bayes import MultinomialNB

m2=MultinomialNB()
m2.fit(x_train_vec,y_train)
y_pred=m2.predict(X_test_vec)

In [93]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.4062171132518417
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.89      0.31      0.47        54
            Arts & Entertainment/Music       0.94      0.21      0.34        77
          Autos & Vehicles/Maintenance       0.52      0.21      0.30        62
                Autos & Vehicles/Sales       0.22      0.47      0.30        66
            Beauty & Fitness/Cosmetics       0.63      0.53      0.57        55
     Beauty & Fitness/Fitness Training       0.89      0.28      0.43        60
                  Billing and Payments       0.87      0.56      0.68       990
            Books & Literature/Fiction       0.51      0.44      0.47        52
        Books & Literature/Non-Fiction       0.48      0.62      0.54        63
   Business & Industrial/Manufacturing       0.58      0.56      0.57        73
                      Customer Service       0.27      0.45      0

In [94]:
from sklearn.svm import LinearSVC

m3=LinearSVC()
m3.fit(x_train_vec,y_train)
y_pred=m3.predict(X_test_vec)

In [95]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.5028737958390674
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.79      0.63      0.70        54
            Arts & Entertainment/Music       0.68      0.61      0.64        77
          Autos & Vehicles/Maintenance       0.47      0.47      0.47        62
                Autos & Vehicles/Sales       0.44      0.47      0.46        66
            Beauty & Fitness/Cosmetics       0.73      0.80      0.77        55
     Beauty & Fitness/Fitness Training       0.60      0.62      0.61        60
                  Billing and Payments       0.77      0.68      0.72       990
            Books & Literature/Fiction       0.57      0.63      0.60        52
        Books & Literature/Non-Fiction       0.75      0.71      0.73        63
   Business & Industrial/Manufacturing       0.72      0.71      0.72        73
                      Customer Service       0.37      0.37      0

In [96]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

In [97]:
from sklearn.ensemble import RandomForestClassifier

m4=RandomForestClassifier(class_weight=class_weights)
m4.fit(x_train_vec,y_train)
y_pred=m4.predict(X_test_vec)

In [98]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.6968347769772525
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.91      0.74      0.82        54
            Arts & Entertainment/Music       0.86      0.62      0.72        77
          Autos & Vehicles/Maintenance       0.59      0.58      0.59        62
                Autos & Vehicles/Sales       0.47      0.56      0.51        66
            Beauty & Fitness/Cosmetics       0.76      0.87      0.81        55
     Beauty & Fitness/Fitness Training       0.71      0.62      0.66        60
                  Billing and Payments       0.93      0.77      0.84       990
            Books & Literature/Fiction       0.62      0.69      0.65        52
        Books & Literature/Non-Fiction       0.73      0.78      0.75        63
   Business & Industrial/Manufacturing       0.65      0.70      0.68        73
                      Customer Service       0.67      0.67      0

In [ ]:
import joblib

joblib.dump((m4, tfidf), "model1.pkl", compress=9)

In [56]:
from sklearn.ensemble import RandomForestClassifier

# m4 = RandomForestClassifier(
#     class_weight=class_weights,
#     random_state=42,
#     n_jobs=-1
# )
TfidfVectorizer(
    max_features=2500,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

m4.fit(x_train_vec, y_train)
y_pred=m4.predict(X_test_vec)

In [57]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.6424350360236379
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.52      0.46      0.49        54
            Arts & Entertainment/Music       0.78      0.49      0.60        77
          Autos & Vehicles/Maintenance       0.42      0.42      0.42        62
                Autos & Vehicles/Sales       0.35      0.45      0.39        66
            Beauty & Fitness/Cosmetics       0.65      0.67      0.66        55
     Beauty & Fitness/Fitness Training       0.63      0.52      0.57        60
                  Billing and Payments       0.90      0.73      0.81       990
            Books & Literature/Fiction       0.43      0.48      0.45        52
        Books & Literature/Non-Fiction       0.41      0.52      0.46        63
   Business & Industrial/Manufacturing       0.49      0.56      0.52        73
                      Customer Service       0.64      0.65      0

In [59]:
import pickle

with open("model1.pkl","wb") as f:
  pickle.dump((m4,tfidf),f)

In [ ]:
text=df['tokens']

In [ ]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
tokenizer = Tokenizer(num_words=20000,oov_token="<OOV>")
tokenizer.fit_on_texts(text)
sequences = tokenizer.texts_to_sequences(text)
pad=pad_sequences(sequences,maxlen=150,padding='post',truncating='post')

from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
y=le.fit_transform(df['queue'])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    pad,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional,Dropout
from tensorflow.keras.optimizers import Adam # Import Adam optimizer

num_classes=len(set(y))
model=Sequential()

# Set input_length to 50 to match maxlen used in pad_sequences
model.add(Embedding(input_dim=5000+1,output_dim=150,input_length=60))
model.add(Bidirectional(LSTM(128, return_sequences=False)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dense(num_classes,activation='softmax'))

# Use 'sparse_categorical_crossentropy' with integer labels
model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=0.001),metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=1,
    min_lr=1e-5
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_lstm),
    y=y_train_lstm
)

class_weights = dict(zip(np.unique(y_train_lstm), class_weights))


In [ ]:
model.fit(
    x_train_lstm,
    y_train_lstm,
    epochs=10,
    batch_size=16,
    validation_data=(x_test_lstm, y_test_lstm),
    callbacks=[early_stop, lr_reduce],
    # class_weight=class_weights
)

In [ ]:
model.save("lstm_model.keras")

In [ ]:
import pickle

with open("lstm.pkl","wb") as f:
  pickle.dump(tokenizer, f)

In [ ]:
model.summary()

In [ ]:
from sklearn.metrics import accuracy_score,classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
df['tokens'] = df['tokens'].apply(lambda x: x.split())

In [ ]:
print(df['tokens'].head())

In [ ]:
unique_words = set()

for tokens in df['tokens']:
    unique_words.update(tokens)

print("Total Unique Words:", len(unique_words))

In [ ]:
from collections import Counter

counter = Counter()

for tokens in df['tokens']:
    counter.update(tokens)

# Vocabulary dictionary
vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}

# Add words to vocab
for word, freq in counter.items():

    # Keep words appearing at least 2 times
    if freq >= 2:
        vocab[word] = len(vocab)

vocab_size = len(vocab)

print("Vocabulary Size:", vocab_size)

In [ ]:
!pip install datasets nltk scikit-learn torch

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from collections import Counter

In [ ]:
class LSTMClassifier(nn.Module):

    def __init__(self,
                 vocab_size,
                 embedding_dim,
                 hidden_dim,
                 output_dim,
                 num_layers=2,
                 dropout=0.3):

        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        embedded = self.embedding(x)

        lstm_out, (hidden, cell) = self.lstm(embedded)

        # Concatenate forward + backward hidden states
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)

        hidden = self.dropout(hidden)

        output = self.fc(hidden)

        return output

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class TicketDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = TicketDataset(X_train, y_train)

test_dataset = TicketDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

In [ ]:
num_classes = len(le.classes_)

print(num_classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes=52

model = LSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=128,
    output_dim=num_classes
)

model = model.to(device)

print(model)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to tensor
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

# Loss function with class weights
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 18

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for texts, labels in train_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

In [ ]:
model.eval()

predictions = []
actuals = []

with torch.no_grad():

    for texts, labels in test_loader:

        texts = texts.to(device)

        outputs = model(texts)

        _, preds = torch.max(outputs, 1)

        predictions.extend(preds.cpu().numpy())

        actuals.extend(labels.numpy())

In [ ]:
accuracy = accuracy_score(actuals, predictions)

print("Accuracy:", accuracy)

print(classification_report(actuals, predictions))